# Road Segment Severity Risk — 4 User Inputs

Notebook สำหรับ Google Colab เพื่อประเมินว่า **หากเกิดอุบัติเหตุภายใต้เงื่อนไขที่ระบุ เหตุนั้นมีความเสี่ยงที่จะมีผู้เสียชีวิตเท่าไร**

ผู้ใช้กรอก Input เพียง 4 ช่อง:

1. รหัสสายทาง
2. กิโลเมตร
3. วันที่และเวลา
4. สภาพอากาศ

ระบบจะแปลงวันที่และเวลาเป็นเดือน วันในสัปดาห์ และชั่วโมงให้อัตโนมัติ ก่อนส่งให้โมเดล

ผลลัพธ์หมายถึง **หากเกิดอุบัติเหตุที่ช่วงถนนและเวลานี้ มีความเสี่ยงเป็นเหตุเสียชีวิตหรือไม่** ไม่ใช่โอกาสที่จะเกิดอุบัติเหตุ

โมเดลไม่ได้ทำนายว่าอุบัติเหตุจะเกิดหรือไม่ เพราะชุดข้อมูลมีเฉพาะเหตุการณ์ที่เกิดอุบัติเหตุแล้ว


## 0. ติดตั้งไลบรารีที่จำเป็น

เซลล์นี้จะติดตั้งเฉพาะแพ็กเกจที่ยังไม่มี โดยใช้ Python Kernel ตัวเดียวกับ Notebook


In [1]:
import importlib.util
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'openpyxl': 'openpyxl',
    'joblib': 'joblib',
    'jinja2': 'jinja2',
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    command = [
        sys.executable, '-m', 'pip', 'install',
        '--quiet', '--disable-pip-version-check',
        *missing_packages,
    ]
    completed = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            'ติดตั้งไลบรารีไม่สำเร็จ:\n' + completed.stderr.strip()
        )
    importlib.invalidate_caches()

still_missing = [
    module_name
    for module_name in required_packages
    if importlib.util.find_spec(module_name) is None
]
if still_missing:
    raise ImportError(f'ยังไม่พบไลบรารีหลังติดตั้ง: {still_missing}')

print('พร้อมใช้งาน: ติดตั้งและตรวจสอบไลบรารีที่จำเป็นครบแล้ว')


พร้อมใช้งาน: ติดตั้งและตรวจสอบไลบรารีที่จำเป็นครบแล้ว


## 1. อัปโหลดข้อมูล

หากเปิดใน Google Colab ให้กด Run แล้วเลือกไฟล์พร้อมกัน 4 ไฟล์  
หากเปิดใน VS Code/Jupyter ให้นำไฟล์ทั้ง 4 ไว้ในโฟลเดอร์เดียวกับ Notebook:

- `accident2023.csv` หรือ `.xlsx`
- `accident2024.csv` หรือ `.xlsx`
- `accident2025.csv` หรือ `.xlsx`
- `accident2026.csv` หรือ `.xlsx`


In [2]:
from pathlib import Path

file_candidates = {
    year: (f'accident{year}.csv', f'accident{year}.xlsx')
    for year in (2023, 2024, 2025, 2026)
}

try:
    from google.colab import files
    print('ตรวจพบ Google Colab — กรุณาเลือกไฟล์ข้อมูลทั้ง 4 ไฟล์')
    uploaded = files.upload()
    available_files = set(uploaded)
except ImportError:
    print('ตรวจพบ VS Code/Jupyter — จะอ่านไฟล์จากโฟลเดอร์ปัจจุบัน')
    available_files = {path.name for path in Path('.').iterdir() if path.is_file()}

selected_files = {}
missing_years = []
for year, candidates in file_candidates.items():
    matched = next((name for name in candidates if name in available_files), None)
    if matched is None:
        missing_years.append(year)
    else:
        selected_files[year] = matched

if missing_years:
    raise FileNotFoundError(
        f'ไม่พบไฟล์ข้อมูลปี: {missing_years}\n'
        'หากใช้ VS Code/Jupyter ให้นำไฟล์ไปไว้ในโฟลเดอร์เดียวกับ Notebook'
    )
print('พบไฟล์ข้อมูลครบแล้ว:', selected_files)


ตรวจพบ VS Code/Jupyter — จะอ่านไฟล์จากโฟลเดอร์ปัจจุบัน
พบไฟล์ข้อมูลครบแล้ว: {2023: 'accident2023.csv', 2024: 'accident2024.csv', 2025: 'accident2025.xlsx', 2026: 'accident2026.xlsx'}


## 2. เตรียมข้อมูลและสร้าง Target


In [3]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FEATURE_ALIASES = {
    'วันที่เกิดเหตุ': ['วันที่เกิดเหตุ', 'วันที่เกิดอุบัติเหตุ', 'วันที่'],
    'เวลา': ['เวลา', 'เวลาเกิดเหตุ', 'เวลาที่เกิดเหตุ'],
    'จำนวนผู้เสียชีวิต': ['จำนวนผู้เสียชีวิต', 'ผู้เสียชีวิต'],
    'รหัสสายทาง': ['รหัสสายทาง', 'รหัสถนน', 'หมายเลขทางหลวง', 'หมายเลขสายทาง'],
    'ก.ม.': ['ก.ม.', 'กม.', 'กิโลเมตร', 'หลักกิโลเมตร', 'KM'],
    'สภาพอากาศ': ['สภาพอากาศ', 'สภาพอากาศขณะเกิดเหตุ', 'อากาศ'],
}

def harmonize_columns(data, year):
    data = data.copy()
    data.columns = [re.sub(r'\s+', ' ', str(column).replace('\ufeff', '')).strip() for column in data.columns]

    for target, aliases in FEATURE_ALIASES.items():
        available = [column for column in aliases if column in data.columns]
        if not available:
            data[target] = pd.NA
            continue

        combined = data[available[0]].replace(r'^\s*$', pd.NA, regex=True)
        for column in available[1:]:
            combined = combined.fillna(data[column].replace(r'^\s*$', pd.NA, regex=True))
        data[target] = combined

    essential = ['วันที่เกิดเหตุ', 'เวลา', 'จำนวนผู้เสียชีวิต']
    unavailable = [column for column in essential if data[column].isna().all()]
    if unavailable:
        raise ValueError(f'ข้อมูลปี {year} ไม่มีคอลัมน์ที่จำเป็น: {unavailable}')
    return data

def parse_date(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors='coerce')
    numeric = pd.to_numeric(series, errors='coerce')
    text_dates = pd.to_datetime(series.where(numeric.isna()), errors='coerce')
    excel_dates = pd.to_datetime(numeric, unit='D', origin='1899-12-30', errors='coerce')
    return text_dates.fillna(excel_dates)

def parse_hour(value):
    if pd.isna(value):
        return np.nan
    if hasattr(value, 'hour'):
        return value.hour + value.minute / 60
    if isinstance(value, (float, int, np.number)):
        return float(value) * 24 if 0 <= float(value) < 1 else float(value)
    try:
        parts = str(value).split(':')
        return int(parts[0]) + int(parts[1]) / 60
    except Exception:
        return np.nan

frames = []
for year in (2023, 2024, 2025, 2026):
    filename = selected_files[year]
    data = pd.read_csv(filename, encoding='utf-8-sig') if filename.endswith('.csv') else pd.read_excel(filename)
    data = harmonize_columns(data, year)
    data['date'] = parse_date(data['วันที่เกิดเหตุ'])
    data['hour'] = data['เวลา'].map(parse_hour)
    data['month'] = data['date'].dt.month
    data['weekday'] = data['date'].dt.dayofweek
    data['year'] = year
    frames.append(data)

df = pd.concat(frames, ignore_index=True)

CATEGORICAL_FEATURES = [
    'รหัสสายทาง',
    'สภาพอากาศ',
]
NUMERIC_FEATURES = ['ก.ม.', 'month', 'weekday', 'hour']
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for column in CATEGORICAL_FEATURES:
    df[column] = df[column].replace(r'^\s*$', pd.NA, regex=True).fillna('ไม่ระบุ').astype(str).str.strip()
df['รหัสสายทาง'] = df['รหัสสายทาง'].str.replace(r'^(\d+)\.0$', r'\1', regex=True)
for column in NUMERIC_FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df['death_count'] = pd.to_numeric(df['จำนวนผู้เสียชีวิต'], errors='coerce')
df = df.dropna(subset=['date', 'death_count']).copy()
df['fatal'] = (df['death_count'] > 0).astype(int)

print(f'ข้อมูลทั้งหมด: {len(df):,} เหตุการณ์')
print(df.groupby('year')['fatal'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'จำนวนเหตุการณ์', 'sum': 'เหตุเสียชีวิต', 'mean': 'สัดส่วนเสียชีวิต'}
))


ข้อมูลทั้งหมด: 89,956 เหตุการณ์
      จำนวนเหตุการณ์  เหตุเสียชีวิต  สัดส่วนเสียชีวิต
year                                                 
2023           23841           2660          0.111573
2024           23952           2554          0.106630
2025           23715           2387          0.100654
2026           18448           1823          0.098818


## 3. ฝึกและประเมินโมเดลแบบข้ามปี

- Train: ปี 2023–2024
- Validation และเลือก Threshold: ปี 2025
- Test: ปี 2026
- `class_weight='balanced'` ช่วยลดผลกระทบจากจำนวนเหตุเสียชีวิตที่น้อยกว่าคลาสไม่เสียชีวิต
- เลือก Threshold โดยตั้งเป้า Precision อย่างน้อย 30% แล้วเลือกจุดที่ Recall สูงที่สุด


In [4]:
preprocessor = ColumnTransformer([
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=5)),
    ]), CATEGORICAL_FEATURES),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler(with_mean=False)),
    ]), NUMERIC_FEATURES),
])

models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(
        max_iter=1200, class_weight='balanced', solver='liblinear', random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=20, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=180, min_samples_leaf=3,
        class_weight='balanced_subsample', n_jobs=-1, random_state=42
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=180, min_samples_leaf=3,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
}

train_mask = df['year'] <= 2024
validation_mask = df['year'] == 2025
test_mask = df['year'] == 2026
X_train = df.loc[train_mask, FEATURES]
X_validation = df.loc[validation_mask, FEATURES]
X_test = df.loc[test_mask, FEATURES]
y_train = df.loc[train_mask, 'fatal']
y_validation = df.loc[validation_mask, 'fatal']
y_test = df.loc[test_mask, 'fatal']

fitted_models = {}
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([('preprocess', clone(preprocessor)), ('model', clone(estimator))])
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    probability = pipeline.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, prediction),
        'Balanced Accuracy': balanced_accuracy_score(y_test, prediction),
        'Precision เสียชีวิต': precision_score(y_test, prediction, zero_division=0),
        'Recall เสียชีวิต': recall_score(y_test, prediction),
        'F1 เสียชีวิต': f1_score(y_test, prediction),
        'ROC AUC': roc_auc_score(y_test, probability),
        'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
    })
    fitted_models[name] = pipeline

# เลือก Threshold ของ Logistic Regression จาก Validation ปี 2025 เท่านั้น
risk_model = fitted_models['Logistic Regression']
validation_probability = risk_model.predict_proba(X_validation)[:, 1]
validation_precision, validation_recall, candidate_thresholds = precision_recall_curve(
    y_validation, validation_probability
)

target_precision = 0.30
valid_indices = np.where(validation_precision[:-1] >= target_precision)[0]
if len(valid_indices) > 0:
    selected_index = valid_indices[np.argmax(validation_recall[:-1][valid_indices])]
else:
    validation_f1 = (
        2 * validation_precision[:-1] * validation_recall[:-1]
        / (validation_precision[:-1] + validation_recall[:-1] + 1e-10)
    )
    selected_index = int(np.argmax(validation_f1))

selected_threshold = float(candidate_thresholds[selected_index])
test_probability = risk_model.predict_proba(X_test)[:, 1]
tuned_prediction = (test_probability >= selected_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, tuned_prediction, labels=[0, 1]).ravel()
rows.append({
    'Model': 'Logistic Regression (Threshold tuned)',
    'Accuracy': accuracy_score(y_test, tuned_prediction),
    'Balanced Accuracy': balanced_accuracy_score(y_test, tuned_prediction),
    'Precision เสียชีวิต': precision_score(y_test, tuned_prediction, zero_division=0),
    'Recall เสียชีวิต': recall_score(y_test, tuned_prediction),
    'F1 เสียชีวิต': f1_score(y_test, tuned_prediction),
    'ROC AUC': roc_auc_score(y_test, test_probability),
    'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
})

results = pd.DataFrame(rows).sort_values('Balanced Accuracy', ascending=False)

# แสดงผลโดยไม่ใช้ DataFrame.style เพื่อให้ทำงานได้แม้ไม่มี Jinja2
results_display = results.copy()
percentage_columns = [
    'Accuracy',
    'Balanced Accuracy',
    'Precision เสียชีวิต',
    'Recall เสียชีวิต',
    'F1 เสียชีวิต',
    'ROC AUC',
]
for column in percentage_columns:
    results_display[column] = results_display[column].map(lambda value: f'{value:.1%}')
display(results_display.reset_index(drop=True))
print(f'Threshold ที่เลือกจากปี 2025: {selected_threshold:.3f}')
print(f'Validation Precision: {validation_precision[selected_index]:.1%}')
print(f'Validation Recall: {validation_recall[selected_index]:.1%}')


,Model,Accuracy,Balanced Accuracy,Precision เสียชีวิต,Recall เสียชีวิต,F1 เสียชีวิต,ROC AUC,TP,FN,FP,TN
0,Logistic Regression,57.8%,61.0%,14.2%,65.1%,23.3%,66.5%,1186,637,7150,9475
1,Extra Trees,67.6%,60.5%,15.6%,51.6%,23.9%,67.3%,941,882,5095,11530
2,Decision Tree,35.6%,60.2%,12.4%,90.9%,21.8%,63.5%,1657,166,11722,4903
3,Random Forest,74.7%,58.7%,16.6%,38.8%,23.2%,67.8%,707,1116,3559,13066
4,Logistic Regression (Threshold tuned),89.4%,51.5%,27.2%,4.2%,7.2%,66.5%,76,1747,203,16422
5,Baseline,90.1%,50.0%,0.0%,0.0%,0.0%,50.0%,0,1823,0,16625


Threshold ที่เลือกจากปี 2025: 0.767
Validation Precision: 30.0%
Validation Recall: 4.8%


### แนวทางเลือกโมเดล

- หากต้องการ Accuracy โดยรวมสูง: เลือก **Random Forest**
- หากต้องการตรวจจับเหตุที่เสี่ยงเสียชีวิตให้ได้มากขึ้น: เลือก **Logistic Regression**

Baseline อาจมี Accuracy สูงเพราะทำนายว่าไม่เสียชีวิตทุกเหตุการณ์ แต่ Recall ของเหตุเสียชีวิตเป็น 0% จึงไม่ควรเลือกใช้งาน


## 4. กรอกค่าแล้วรันเพื่อ Predict


In [5]:
# แก้เฉพาะค่า 4 บรรทัดนี้ แล้วกด Run เซลล์
route_code = '6'
kilometer = 139.7
event_datetime = '2026-07-18 23:00'
weather = 'ฝนตก'

event_dt = pd.to_datetime(event_datetime, errors='coerce')
if pd.isna(event_dt):
    raise ValueError('กรุณากรอกวันที่และเวลาเป็น YYYY-MM-DD HH:MM')

known_routes = set(X_train['รหัสสายทาง'].dropna().astype(str))
known_weather = set(X_train['สภาพอากาศ'].dropna().astype(str))
if route_code not in known_routes:
    raise ValueError(f'ไม่พบรหัสสายทาง {route_code} ในชุดฝึก')
if weather not in known_weather:
    raise ValueError(f'ไม่พบสภาพอากาศ {weather} ในชุดฝึก')

new_event = pd.DataFrame([{
    'รหัสสายทาง': route_code,
    'สภาพอากาศ': weather,
    'ก.ม.': float(kilometer),
    'month': int(event_dt.month),
    'weekday': int(event_dt.dayofweek),
    'hour': float(event_dt.hour + event_dt.minute / 60),
}])[FEATURES]

risk_model = fitted_models['Logistic Regression']
fatal_score = float(risk_model.predict_proba(new_event)[0, 1])
predicted_class = int(fatal_score >= selected_threshold)
result_label = 'เสี่ยงรุนแรง' if predicted_class == 1 else 'ความเสี่ยงทั่วไป'

segment_start = int(np.floor(float(kilometer)))
segment_end = segment_start + 1
history = df[
    (df['year'] <= 2025)
    & (df['รหัสสายทาง'] == route_code)
    & (df['ก.ม.'] >= segment_start)
    & (df['ก.ม.'] < segment_end)
]
historical_accidents = int(len(history))
historical_fatal = int(history['fatal'].sum())
historical_fatal_rate = historical_fatal / historical_accidents if historical_accidents else 0.0

print(f'ถนนสาย {route_code} กม. {segment_start}–{segment_end}')
print(f'เกิดอุบัติเหตุย้อนหลัง {historical_accidents:,} ครั้ง')
print(f'เป็นเหตุเสียชีวิต {historical_fatal:,} ครั้ง')
print(f'อัตราเหตุเสียชีวิต {historical_fatal_rate:.1%}')
print(f'วันที่และเวลา: {event_dt:%Y-%m-%d %H:%M}')
print(f'สภาพอากาศ: {weather}')
print(f'คะแนนความเสี่ยงรุนแรง: {fatal_score:.1%}')
print(f'Threshold ที่ใช้: {selected_threshold:.1%}')
print(f'ผลประเมิน: {result_label}')


ถนนสาย 6 กม. 139–140
เกิดอุบัติเหตุย้อนหลัง 2 ครั้ง
เป็นเหตุเสียชีวิต 0 ครั้ง
อัตราเหตุเสียชีวิต 0.0%
วันที่และเวลา: 2026-07-18 23:00
สภาพอากาศ: ฝนตก
คะแนนความเสี่ยงรุนแรง: 9.3%
Threshold ที่ใช้: 76.7%
ผลประเมิน: ความเสี่ยงทั่วไป


## 5. บันทึกโมเดลเพื่อนำไปใช้ต่อ

ดาวน์โหลดไฟล์โมเดลจาก Colab ได้หลังรันเซลล์นี้


In [6]:
import joblib

MODEL_TO_SAVE = 'Logistic Regression'
model_filename = 'accident_road_risk_4_inputs.joblib'
model_artifact = {
    'model': fitted_models[MODEL_TO_SAVE],
    'threshold': selected_threshold,
    'features': FEATURES,
    'segment_size_km': 1,
}
joblib.dump(model_artifact, model_filename)
print(f'บันทึกโมเดลแล้ว: {model_filename}')

try:
    from google.colab import files
    files.download(model_filename)
except ImportError:
    print('ไฟล์โมเดลอยู่ในโฟลเดอร์เดียวกับ Notebook')


บันทึกโมเดลแล้ว: accident_road_risk_4_inputs.joblib
ไฟล์โมเดลอยู่ในโฟลเดอร์เดียวกับ Notebook


## ข้อควรระวัง

- Probability เป็นคะแนนความเสี่ยงจากรูปแบบในข้อมูล ไม่ใช่การรับรองว่าจะเสียชีวิตจริง
- ค่าใหม่ที่ไม่เคยปรากฏในชุดฝึก เช่น ชื่อประเภทสายทางที่สะกดต่างกัน จะถูกจัดการได้ แต่ควรใช้คำให้ตรงกับข้อมูลเดิม
- ควรใช้ผลเพื่อช่วยคัดกรองหรือจัดลำดับความเสี่ยง ไม่ควรใช้แทนการตัดสินใจของเจ้าหน้าที่
